# Part 1 — What is a Network, and What is a Knowledge Graph?

This notebook covers the vocabulary for the rest of the day. It is deliberately
light on code: the point is to be able to look at a biological question and say
what the nodes are, what the edges are, and *where the edges came from* — because
that last question turns out to matter more than anything else.

We then meet the specific knowledge graph used in Sessions 1, 3 and 4: a
breast-cancer-focused graph built from [Open Targets](https://platform.opentargets.org/)
and the [MONDO Disease Ontology](https://mondo.monarchinitiative.org/).

## Learning objectives

By the end of this notebook we will be able to:

- Name the parts of a graph: vertex, edge, degree, path, component.
- Distinguish a **directed** from an **undirected** graph, and say why it matters here.
- Build a small graph by hand in NetworkX and inspect it.
- Explain the difference between an **inferred** network and a **curated** knowledge graph.
- Say what makes a graph a *knowledge* graph: typed nodes, typed edges, and provenance.
- Describe where our data comes from and under what licence.

## 1. Networks, in the smallest possible terms

A **network** (or **graph**) is two things:

- a set of **vertices** (also called **nodes**) — the objects
- a set of **edges** — the pairs of objects that are related

That is all. Everything else is vocabulary built on top:

| Term | Meaning |
|---|---|
| **Degree** | How many edges a node has. A high-degree node is a **hub**. |
| **Path** | A sequence of edges we can follow from one node to another. |
| **Component** | A group of nodes all reachable from each other. |
| **Self-loop** | An edge from a node to itself. |
| **Isolated vertex** | A node with no edges at all. |
| **Directed** | Edges have a direction (A → B is not B → A). |
| **Undirected** | Edges are symmetric (A — B). |

![Graph terminology](images/vertex_types.png)

*Diagram reused from earlier gene co-expression network teaching material.*

### Why directedness matters for us

It is tempting to treat this as a technicality. It is not. Our graph contains edges like:

> `triple-negative breast carcinoma` **is_a** `breast carcinoma`

That is emphatically *not* symmetric — every triple-negative breast carcinoma is a
breast carcinoma, but not the reverse. If we throw the direction away we can still
draw the graph, but we can no longer *climb* it, and climbing it is exactly what we
do in Part 3.

We will mostly work **undirected**, because degree, hubs and components are more
intuitive that way, and keep the direction as an edge attribute so nothing is lost.

## 2. Building a graph by hand

Before loading anything real, here is the whole NetworkX API we need.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import networkx as nx
import matplotlib.pyplot as plt

# A graph starts empty.
G = nx.Graph()

# Nodes can carry arbitrary attributes - here, what kind of thing the node is.
G.add_node("BRCA1", type="gene")
G.add_node("BRCA2", type="gene")
G.add_node("breast cancer", type="disease")
G.add_node("ovarian cancer", type="disease")

# Edges can carry attributes too. `weight` is just a name; it has no special
# meaning to NetworkX beyond a few algorithms that look for it by default.
G.add_edge("BRCA1", "breast cancer", type="associated_with", weight=0.92)
G.add_edge("BRCA2", "breast cancer", type="associated_with", weight=0.89)
G.add_edge("BRCA1", "ovarian cancer", type="associated_with", weight=0.85)

print(G)

Graph with 4 nodes and 3 edges


In [2]:
# Degree: how many edges each node has.
for node, degree in sorted(G.degree(), key=lambda x: -x[1]):
    print(f"{node:16s} degree={degree}  type={G.nodes[node]['type']}")

BRCA1            degree=2  type=gene
breast cancer    degree=2  type=disease
BRCA2            degree=1  type=gene
ovarian cancer   degree=1  type=disease


Notice that `ovarian cancer` has degree 1 and `BRCA1` has degree 2. In a graph this
small that is trivia. In a graph of 100,000 nodes, degree is the first thing we
look at — it tells us which entities the underlying literature has paid attention
to. Keep that phrasing in mind: **degree measures attention, not importance.**

In [3]:
# Neighbours and paths - the two operations that make a graph useful.
print("Neighbours of BRCA1:", list(G.neighbors("BRCA1")))
print("Path from ovarian cancer to BRCA2:",
      nx.shortest_path(G, "ovarian cancer", "BRCA2"))

Neighbours of BRCA1: ['breast cancer', 'ovarian cancer']
Path from ovarian cancer to BRCA2: ['ovarian cancer', 'BRCA1', 'breast cancer', 'BRCA2']


That path — `ovarian cancer → BRCA1 → breast cancer → BRCA2` — is a *tiny* example
of the thing this whole day is building towards. Nobody recorded a fact linking
ovarian cancer to BRCA2 directly. The graph produced it by composition.

That is what a knowledge graph is *for*: answering questions nobody explicitly
stored the answer to.

## 3. Two very different ways to get a network

Both of the pictures below are networks. They are built from opposite directions,
and confusing them is the most common mistake in this field.

![Inferred vs curated](images/curated_vs_inferred.png)

**Inferred networks** are *computed from measurements*. In gene co-expression work —
including Session 2 of this workshop — we take a gene expression matrix, correlate
every gene against every other gene, and draw an edge wherever the correlation
clears a threshold. The edges are a **statistical claim about our data**.

- One node type (genes, or patients).
- Edge weights are correlations.
- Change the threshold and we get a different network. There is no "true" one.
- The network can contain relationships nobody has ever described before. That is
  the point of building it — it is a hypothesis generator.

**Curated knowledge graphs** are *read out of a database of recorded facts*. Somebody
asserted that this gene is associated with that disease, and the graph records it.

- Several node types (genes, diseases, drugs, phenotypes, codes...).
- Edges are typed: *this kind* of relationship, not just "related".
- Edges carry **provenance**: which study, which database, how much evidence.
- The graph can only contain what somebody already knew. It is a knowledge
  *retrieval* structure, not a discovery one.

### Who is the "somebody"?

Usually a **curator** — a biologist employed by a database to read the published
literature and turn its findings into structured entries a computer can use.
Reading a paper that reports a BRCA1 mutation in a breast cancer family, and
writing a row that says *gene BRCA1, disease breast cancer, evidence PMID 12345*,
is curation. The word covers automated sources too: text-mining pipelines and
genome-wide association studies contribute entries the same way.

The consequence matters more than the definition. **Every edge exists because a
person or a pipeline put it there** — which is why a knowledge graph can only ever
tell us what has already been written down.

<details>
<summary><b>Why does this distinction matter so much?</b> (click to expand)</summary>

Because the two answer different questions, and the failure modes are opposite.

If we ask an **inferred** network "are these two genes related?" and it says yes,
the risk is that the correlation is an artefact — batch effect, cell-type
composition, a confounder.

If we ask a **curated** knowledge graph the same question and it says *no*, the
risk is completely different: the absence of an edge does not mean the absence of
a relationship. It very often means **nobody has looked yet**.

We see this concretely in Part 2, where the disease with the most genes in
our graph is not the most important disease — it is the most *studied* one. And in
Part 3, where a major breast cancer subtype turns out to have no gene associations
at all, not because it has no genes, but because the evidence was filed under a
different name.

An inferred network is biased towards what we measured. A knowledge graph is
biased towards what somebody published. Neither bias is fixable by better code.
</details>

## 4. What makes it a *knowledge* graph

A plain graph says *A relates to B*. A knowledge graph says **who** A and B are,
**how** they relate, and **on what basis** we should believe it.

In practice that means three things:

1. **Typed nodes** — every node declares what kind of entity it is.
2. **Typed edges** — every edge declares what kind of relationship it is.
3. **Provenance** — the edge carries evidence: a score, a source, a count.

That third one is easy to nod along to and hard to take seriously. We take it
seriously in Part 3, where the *kind* of evidence behind an edge turns out to
decide whether the edge means what we think it means.

Here is the schema of the graph we will use all day:

![Knowledge graph schema](images/kg_schema.png)

Three node types and three edge types:

| Edge type | From → To | Meaning | Weighted? |
|---|---|---|---|
| `associated_with` | gene → disease | This gene is implicated in this disease | Yes — an Open Targets association score, 0–1 |
| `is_a` | disease → disease | This disease is a subtype of that one | No |
| `maps_to` | disease → icd10 | This disease corresponds to this clinical billing code | No |

The `is_a` edges are what make this an **ontology**-backed graph rather than a flat
one, and they do a lot of work in Part 3.

## 5. Where the data comes from

Two public sources, both redistributable, which is why they can ship inside this
teaching repository.

### [Open Targets Platform](https://platform.opentargets.org/)

A drug-target discovery platform that aggregates gene–disease evidence from
genetics, somatic mutations, drugs, pathways, RNA expression and the literature.
We take three things from it:

- **Gene entities** — Ensembl gene IDs and their approved symbols.
- **Disease entities and the ontology hierarchy** — the `is_a` structure.
- **Association scores** — a 0–1 summary of how much evidence links a gene to a disease.

Licence: [CC0 1.0](https://platform-docs.opentargets.org/licence) — public domain.
Data downloads: <https://platform.opentargets.org/downloads>

### [MONDO Disease Ontology](https://mondo.monarchinitiative.org/)

A unified disease ontology that merges several older vocabularies and, crucially
for us, records **cross-references** to other coding systems — including ICD-10.

Licence: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) — attribution required.
Mapping file: [`mondo.sssom.tsv`](https://github.com/monarch-initiative/mondo/blob/master/src/ontology/mappings/mondo.sssom.tsv)

### Why not DisGeNET?

It is the obvious third option and it appears in a lot of papers. As of 2024
it moved to commercial licensing: the free academic licence does not permit
downloading the full database, and requires per-user registration with roughly a
week's approval. That makes it impossible to ship in a public teaching repo.

### One design decision worth flagging

Open Targets keys genes by **Ensembl gene ID** (`ENSG00000012048`), which is the
same identifier space as the TCGA-BRCA transcriptomics matrix used in Session 2.

That is not a coincidence — it was chosen for it. It means a gene list produced by
the multi-omics work in Session 2 can be looked up in this graph directly, without
a symbol-mapping step that would silently lose 10–20% of the genes. We use that
bridge at the end of Part 3.

(There is still one gotcha in that join, which we hit and fix in Part 3.)

## 6. The data files

The graph ships as three small CSVs in `data/`, generated by
`data-prep/build_kg_data.py`. That script is a **developer** script — it downloads
about 1.1 GB from Open Targets and cuts it down. We do not need to run it.

| File | Contents |
|---|---|
| `kg_nodes.csv` | `id`, `type`, `name`, `extra` |
| `kg_edges.csv` | `source`, `target`, `type`, `weight`, `evidence` |
| `kg_evidence.csv` | the same gene–disease edges split by *kind* of evidence |
| `icd10_map.csv` | MONDO id → ICD-10 code and label |

A first look:

In [4]:
import pandas as pd

nodes = pd.read_csv("data/kg_nodes.csv")
edges = pd.read_csv("data/kg_edges.csv")

print(f"{len(nodes):,} nodes, {len(edges):,} edges\n")
display(nodes.head())
display(edges.head())

881 nodes, 1,768 edges



,id,type,name,extra
0,EFO_0009443,disease,BRCAX breast cancer,OTAR_0000017; MONDO_0045024
1,EFO_0009649,disease,susceptibility to breast cancer,OTAR_0000017; MONDO_0045024
2,EFO_0009782,disease,progesterone-receptor positive breast cancer,OTAR_0000017; MONDO_0045024
3,EFO_0022984,disease,bilateral breast cancer,OTAR_0000017; MONDO_0045024
4,MONDO_0000552,disease,breast lobular carcinoma,MONDO_0002051; MONDO_0045024; OTAR_0000017


,source,target,type,weight,evidence
0,EFO_0009443,MONDO_0004989,is_a,NaN,NaN
1,EFO_0009649,MONDO_0004989,is_a,NaN,NaN
2,EFO_0009782,MONDO_0004989,is_a,NaN,NaN
3,EFO_0022984,MONDO_0007254,is_a,NaN,NaN
4,MONDO_0000552,MONDO_0004988,is_a,NaN,NaN


In [5]:
# What is actually in there?
print("Node types:")
print(nodes["type"].value_counts().to_string())
print("\nEdge types:")
print(edges["type"].value_counts().to_string())

Node types:


type
gene       760
disease     90
icd10       31

Edge types:
type
associated_with    1656
is_a                 81
maps_to              31


Two things to notice already, both of which we come back to:

- The graph is **mostly genes** (760 of 881 nodes). That is normal for a
  gene–disease graph and it shapes what the visualisations look like.
- There are only **31 ICD-10 nodes** for **90 diseases** — and the gap is not
  spread evenly. Working out where it falls is the first exercise in Part 3.

## Summary

- A graph is nodes plus edges; degree, paths and components are all derived from those.
- **Inferred** networks are computed from measurements; **curated** knowledge graphs
  are read from recorded facts. Their biases are opposite and neither is fixable by code.
- A *knowledge* graph adds typed nodes, typed edges and provenance.
- Ours has 3 node types (gene, disease, icd10) and 3 edge types (`associated_with`,
  `is_a`, `maps_to`), built from Open Targets (CC0) and MONDO (CC BY 4.0).
- Every gene–disease edge also has an evidence breakdown in `kg_evidence.csv`.
- Genes are keyed by Ensembl ID so the graph joins to the Session 2 omics data.

**Next:** in Part 2 we build the graph in NetworkX and look at what it is shaped like.